In [0]:
df_renewal = spark.read.table("workspace.`ds-raw-datasets`.raw_renewal_calls")

df_renewal = df_renewal.select(
    "Call_ID",
    "Call_Direction",
    "Co_Ref",
    "Call_Date",
    "Serious_Complaint",
    "Other_Complaint",
    "Discussion_on_Price_Increase",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Call_Reschedule_Request",
    "Explicit_Competitor_Mention",
    "Analysed_Call"
)

In [0]:
df_renewal.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── 1. Load / prepare billings ──────────────────────────────────────────
df_billings = spark.table("workspace.`ds-raw-datasets`.raw_billings") \
    .filter(F.col("prospect_outcome") != "Open") \
    .withColumn("datediff", F.datediff("closed_date", "prospect_renewal_date")) \
    .filter(F.col("datediff") < 29) \
    .withColumn("index", F.monotonically_increasing_id())

# Use df_renewal instead of spark.table("post_renewal_churn.raw.renewal_calls")
df_calls = df_renewal \
    .filter(F.col("Analysed_Call") == "1") \
    .filter(F.col("Customer_Renewal_Response_Category") != "null") \
    .filter(F.col("Customer_Renewal_Response_Category") != "Not Mentioned")

# ── 3. Join on co_ref — explicit condition to avoid ambiguous reference ──
df_joined = df_billings.join(
    df_calls,
    on=df_billings["Co_Ref"] == df_calls["Co_Ref"],
    how="left"
).filter(
    (F.col("Call_Date") >= F.col("prospect_renewal_date")) &
    (F.col("Call_Date") <= F.col("closed_date"))
).drop(df_calls["Co_Ref"])

# ── 4. Keep only the LATEST call per billing row (index) ────────────────
window = Window.partitionBy("index").orderBy(F.desc("Call_Date"))

df_latest_call = df_joined \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

df_latest_call.display()
df_final = df_latest_call
df_final.write.mode("overwrite").saveAsTable("workspace.`ds-raw-datasets`.joined_two_tables")

In [0]:
df_latest_call.count()

In [0]:


from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from scipy.stats import chi2_contingency, ttest_ind
import pandas as pd

# Columns to test
columns = [
    "Call_ID",
    "Call_Direction",
    "Co_Ref",
    "Call_Date",
    "Serious_Complaint",
    "Other_Complaint",
    "Discussion_on_Price_Increase",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Call_Reschedule_Request",
    "Explicit_Competitor_Mention",
    "Analysed_Call"
]

# Categorical columns for chi-square test
categorical_cols = [
    "Call_Direction",
    "Serious_Complaint",
    "Other_Complaint",
    "Discussion_on_Price_Increase",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Call_Reschedule_Request",
    "Explicit_Competitor_Mention",
    "Analysed_Call"
]

# Numeric columns for t-test (if any)
numeric_cols = []

# Prepare DataFrame for testing
df_test = df_latest_call.select(columns + ["prospect_outcome"])

results = []

for col in columns:
    h0 = f"There is no association between {col} and prospect_outcome."
    h1 = f"There is an association between {col} and prospect_outcome."
    if col in categorical_cols:
        # Chi-square test
        pdf = df_test.groupBy(col, "Prospect_Outcome").count().toPandas().pivot(index=col, columns="Prospect_Outcome", values="count").fillna(0)
        chi2, p, _, _ = chi2_contingency(pdf.values)
        result = "Reject H0" if p < 0.05 else "Fail to reject H0"
        results.append((col, h0, h1, p, result))
    elif col in numeric_cols:
        # T-test
        pdf = df_test.select(col, "Prospect_Outcome").toPandas()
        groups = pdf.groupby("Prospect_Outcome")[col]
        if len(groups) == 2:
            vals1 = groups.get_group(list(groups.groups.keys())[0]).dropna()
            vals2 = groups.get_group(list(groups.groups.keys())[1]).dropna()
            t_stat, p = ttest_ind(vals1, vals2)
            result = "Reject H0" if p < 0.05 else "Fail to reject H0"
            results.append((col, h0, h1, p, result))
        else:
            results.append((col, h0, h1, None, "Not enough groups for t-test"))
    else:
        results.append((col, h0, h1, None, "Not tested"))

# Print results
for col, h0, h1, p, result in results:
    print(f"Column: {col}")
    print(f"H0: {h0}")
    print(f"H1: {h1}")
    print(f"p-value: {p}")
    print(f"Result: {result}\n")

In [0]:
reject_h0_cols = [
    "Co_Ref",
    "Call_ID",
    "Call_Direction",
    "Call_Date",
    "Serious_Complaint",
    "Other_Complaint",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Explicit_Competitor_Mention"
]

df_reject_h0 = df_latest_call.select(reject_h0_cols + ["prospect_outcome"])
display(df_reject_h0)

In [0]:
reject_h0_cols = [
    "Co_Ref",
    "Call_ID",
    "Call_Direction",
    "Call_Date",
    "Serious_Complaint",
    "Other_Complaint",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Explicit_Competitor_Mention"
]



In [0]:
df = spark.read.table("workspace.`ds-raw-datasets`.raw_renewal_calls")

df = df.select(
    *[i.lower() for i in ["Co_Ref",
    "Call_ID",
    "Call_Direction",
    "Call_Date",
    "Serious_Complaint",
    "Other_Complaint",
    "Renewal_Impact_Due_to_Price_Increase",
    "Discount_or_Waiver_Requested",
    "Explicit_Competitor_Mention", "analysed_call"]]
)

In [0]:
df = df.withColumn('serious_complaint', F.when(F.col("serious_complaint").isNull() | (F.col("serious_complaint") == ""), "UnKnown").otherwise(F.col("serious_complaint")))
display(df.select("serious_complaint").distinct())

In [0]:
df = df.withColumn('other_complaint', F.when(F.col("other_complaint").isNull(), "UnKnown").otherwise(F.col("other_complaint")))

In [0]:
display(df.select("other_complaint").distinct())

In [0]:
display(df.select("renewal_impact_due_to_price_increase").distinct())

In [0]:
df = df.withColumn(
    'renewal_impact_due_to_price_increase',
    F.when(F.col("renewal_impact_due_to_price_increase").isNull(), "UnKnown")
     .when(F.col("renewal_impact_due_to_price_increase").rlike(r"(?i)^\s*no\s*$|^\s*\*\*no\*\*\s*$"), "No")
     .when(F.col("renewal_impact_due_to_price_increase").rlike(r"(?i)^\s*yes\s*$|^yes\s*\(.*\)$|^yes,.*$|^yes\s*due.*$"), "Yes")
     .when(F.col("renewal_impact_due_to_price_increase").rlike(r"(?i)^\s*n/a\s*$|^\s*not applicable\s*$"), "UnKnown")
     .when(F.col("renewal_impact_due_to_price_increase").rlike(r"(?i)^\s*\[yes/no\]\s*$"), "Yes/No")
     .otherwise(F.col("renewal_impact_due_to_price_increase"))
)

In [0]:
display(df.select("renewal_impact_due_to_price_increase").distinct())

In [0]:
display(df.select("discount_or_waiver_requested").distinct())

In [0]:
df = df.withColumn(
    'discount_or_waiver_requested',
    F.when(F.col("discount_or_waiver_requested").isNull(), "UnKnown")
     .when(F.col("discount_or_waiver_requested").rlike(r"(?i)^\s*no\s*$|^\s*\*\*no\*\*\s*$|no\s*\(.*\)|no,.*|no\s*but.*|no\s*although.*|no\s*the.*|no\s*replaced.*"), "No")
     .when(F.col("discount_or_waiver_requested").rlike(r"(?i)^\s*yes\s*$|^yes\s*\(.*\)$|^yes,.*$|yes\s*the.*|yes\s*customer.*|yes\s*agent.*"), "Yes")
     .when(F.col("discount_or_waiver_requested").rlike(r"(?i)^\s*n/a\s*$"), "N/A")
     .when(F.col("discount_or_waiver_requested").rlike(r"(?i)^\s*not applicable\s*$"), "Not applicable")
     .when(F.col("discount_or_waiver_requested").rlike(r"(?i)^\s*\[yes/no\]\s*$"), "Yes/No")
     .when(F.col("discount_or_waiver_requested").rlike(r"The customer was not asking for a discount on a price increase, but was instead confirming and proceeding with a previously negotiated price from last year's discussion."), "No")
     .otherwise(F.col("discount_or_waiver_requested"))
)

In [0]:
df = df.withColumn(
    'discount_or_waiver_requested',
    F.when(F.col("discount_or_waiver_requested")=='N/A', "UnKnown").otherwise(F.col("discount_or_waiver_requested"))
)

In [0]:
display(df.select("discount_or_waiver_requested").distinct())

In [0]:

display(df.select("explicit_competitor_mention").distinct())

In [0]:
df = df.withColumn(
    'explicit_competitor_mention',
    F.when(F.col("explicit_competitor_mention").isNull(), "UnKnown")
     .when(F.col("explicit_competitor_mention")=="XXXX", "UnKnown")
     .when(F.col("explicit_competitor_mention")=="Prompt 2:", "UnKnown")
     .when(F.col("explicit_competitor_mention")=="UNAVAILABLE", "UnKnown")
     .when(F.col("explicit_competitor_mention").rlike(r"(?i)^\s*yes\s*$"), "Yes")
     .otherwise("No")
)

display(df.select("explicit_competitor_mention").distinct())

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.`ds-datasets`.renewal_calls")

In [0]:
%sql
select * from workspace.`ds-raw-datasets`.renewal_call_cleaned;